In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

In [2]:
class AgentsState(MessagesState):
    current_agent: str  # agent 이동 정보
    transfered_by: str  # agent 이동 정보


llm = init_chat_model("openai:gpt-5-nano")

In [ ]:
def make_agent(prompt, tools):

    def agent_node(state: AgentsState):
        llm_with_tools = llm.bind_tools(tools)  # llm에 tool binding
        response = llm_with_tools.invoke(
            f"""
            {prompt}

            당신에게는 다른 상담원에게 대화를 전달하는 'handoff_tool'이라는 도구가 있습니다. 이 도구는 다른 상담원에게 전달할 때만 사용하고, 자기 자신에게 전달하는 용도로는 사용하지 마세요.

            Conversation History:
            {state["messages"]}
            """
        )
        return {"messages": [response]}

    agent_builder = StateGraph(AgentsState)

    agent_builder.add_node("agent", agent_node)
    agent_builder.add_node("tools", ToolNode(tools=tools))

    agent_builder.add_edge(START, "agent")
    agent_builder.add_conditional_edges("agent", tools_condition)  # toolnode에 전달
    agent_builder.add_edge("tools", "agent")
    agent_builder.add_edge("agent", END)

    return agent_builder.compile()

In [4]:
@tool
def handoff_tool(transfer_to: str, transfer_by: str):
    """
    다른 상담원에게 연결(Handoff)합니다.
    고객이 이해할 수 없는 언어로 말할 때 이 도구를 사용하세요.

    `transfer_to`에 가능한 값:
    - `korean_agent` (한국어 상담원)
    - `german_agent` (독일어 상담원)
    - `spanish_agent` (스페인어 상담원)

    `transfered_by`에 가능한 값:
    - `korean_agent`
    - `german_agent`
    - `spanish_agent`

    인자(Args):
        transfer_to: 대화를 전달받을 상담원
        transfered_by: 대화를 전달한 상담원
    """
    return Command(
        update={
            "current_agent": transfer_to,
            "transfer_by": transfer_by,
        },
        goto=transfer_to,
        graph=Command.PARENT,  # current graph 말고, 한 단계 위의 graph 에서 찾음
    )

In [5]:
graph_builder = StateGraph(AgentsState)

# same level (no hierachy)
graph_builder.add_node(
    "korean_agent",
    make_agent(
        prompt="당신은 한국어 고객 지원 상담원입니다. 오직 한국어만 말하고 이해할 수 있습니다.",
        tools=[handoff_tool],
    ),
)
graph_builder.add_node(
    "german_agent",
    make_agent(
        prompt="당신은 독일어 고객 지원 상담원입니다. 오직 독일어만 말하고 이해할 수 있습니다.",
        tools=[handoff_tool],
    ),
)
graph_builder.add_node(
    "spanish_agent",
    make_agent(
        prompt="당신은 스페인어 고객 지원 상담원입니다. 오직 스페인어만 말하고 이해할 수 있습니다.",
        tools=[handoff_tool],
    ),
)

graph_builder.add_edge(START, "korean_agent")

graph = graph_builder.compile()

In [6]:
for event in graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hola! Necesito ayuda con mi cuenta.",
            }
        ]
    },
    stream_mode="updates",
):
    print(event)

{'korean_agent': {'current_agent': 'spanish_agent'}}
{'spanish_agent': {'messages': [HumanMessage(content='Hola! Necesito ayuda con mi cuenta.', additional_kwargs={}, response_metadata={}, id='32b53171-8de6-4700-bb1b-64928fbae09c'), AIMessage(content='¡Hola! Con gusto te ayudo con tu cuenta.\n\nPara entender mejor, ¿qué problema específico tienes? Algunas situaciones comunes:\n- No puedo iniciar sesión o necesito restablecer la contraseña.\n- No puedo ver o editar mi información.\n- Hay cargos desconocidos o un problema de pago.\n- Necesito verificar el estado de mi suscripción.\n\nPara avanzar, por favor responde:\n- ¿Cuál es el correo o nombre de usuario asociado a la cuenta?\n- ¿Qué mensaje de error aparece y en qué paso ocurre?\n- ¿Has intentado restablecer la contraseña? ¿Recibiste algún código o correo?\n\nSi prefieres, también puedo transferir la conversación a un agente hispanohablante para una atención más detallada. ¿Quieres que haga la transferencia ahora?', additional_kwarg